# Parallelizarion Workflow :


In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph , START , END
from dotenv import load_dotenv
from typing import TypedDict , Annotated
from pydantic import BaseModel , Field
import operator

In [2]:
load_dotenv()

True

In [3]:
# this model provides by default : Structured Output

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

In [4]:
# Schema : this is used for getting same structured output for any input.

class EvaluationSchema(BaseModel):
    feedback : str = Field(description="Detailed feedback for the essay.")
    score : int = Field(description="Score out of 10" , gt=0 , lt=10)

In [5]:
# Structured output : 

structured_model  = model.with_structured_output(EvaluationSchema)

In [6]:
# Define state : 

class UPSCState(TypedDict):

    essay : str 
    language_feedback : str 
    analysis_feedback : str 
    clarity_feedback  : str 
    overall_feedback  : str 
    individual_scores : Annotated[list[int] , operator.add]  # [8] , [7] , [6] => [8,7,6] : reducer function
    avg_score  : float

In [7]:
# Tasks :

# Task 1 : 

def evaluate_language(state : UPSCState):

    essay = state['essay']
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n{essay}'
    output = structured_model.invoke(prompt)

    return {
        'language_feedback' : output.feedback,
        'individual_scores'  : [output.score]  # return as a list (as defined in the state)
    }


In [8]:
# Task 2 : 

def evaluate_analysis(state : UPSCState):

    essay = state['essay']
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n{essay}'
    output = structured_model.invoke(prompt)

    return {
        'analysis_feedback' : output.feedback,
        'individual_scores'  : [output.score]  # return as a list (as defined in the state)
    }


In [9]:
# Task 3 : 

def evaluate_thought(state : UPSCState):

    essay = state['essay']
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n{essay}'
    output = structured_model.invoke(prompt)

    return {
        'clarity_feedback' : output.feedback,
        'individual_scores'  : [output.score]  # return as a list (as defined in the state)
    }

In [10]:
# Task 4 : 

def final_evaluation(state : UPSCState):

    # summary feedback :

    prompt = f'Based on the following feedback create a summarized feedback \n language feedback - {state['language_feedback']} \n depth of analysis feedback - {state['analysis_feedback']} \n clarity of thought feedback - {state['clarity_feedback']}.'
    overall_feedback = model.invoke(prompt).content

    # average score : 

    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {
        'overall_feedback' : overall_feedback , 'avg_score' : avg_score
    }


In [11]:
# Graph : 

graph = StateGraph(UPSCState)

# Nodes : 

graph.add_node("evaluate_language" , evaluate_language)
graph.add_node("evaluate_analysis" , evaluate_analysis)
graph.add_node("evaluate_thought" , evaluate_thought)
graph.add_node("final_evaluation" , final_evaluation)


# Edges : 

graph.add_edge(START , 'evaluate_language')
graph.add_edge(START , 'evaluate_analysis')
graph.add_edge(START , 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation' , END)


# Compile : 

workflow = graph.compile()


In [12]:
# Evaluate : 

essay = """# Terrorism: A Threat to Global Peace

## Introduction

Terrorism is one of the greatest challenges facing the modern world. It involves the use of violence, intimidation, and fear to achieve political, religious, or ideological objectives. Terrorist attacks often target innocent civilians, causing loss of life, destruction of property, and long-lasting psychological trauma. Beyond immediate harm, terrorism creates fear, disrupts societies, and threatens peace and stability across nations.

## Causes of Terrorism

The causes of terrorism are complex and differ from one situation to another. Factors such as political conflicts, extremist ideologies, social and economic inequalities, discrimination, and regional instability can contribute to the rise of terrorism. In some cases, terrorist organizations exploit grievances, spread propaganda, and recruit vulnerable individuals by promising purpose, identity, or financial support. However, none of these factors justify acts of terrorism or violence against innocent people.

## Impact on Society

The effects of terrorism are far-reaching. The most tragic consequence is the loss of innocent lives and the suffering of victims and their families. Terrorist attacks can also damage infrastructure, disrupt economies, reduce tourism, and increase security costs. In addition, they may create fear, mistrust, and social divisions within communities. Governments often need to strengthen security measures, which can require balancing public safety with the protection of civil liberties.

## Combating Terrorism

Addressing terrorism requires cooperation at local, national, and international levels. Governments strengthen intelligence sharing, law enforcement, border security, and international collaboration to prevent attacks. Education, community engagement, and efforts to counter violent extremist propaganda also play an important role. Promoting dialogue, social inclusion, respect for human rights, and peaceful conflict resolution can help reduce conditions that extremist groups may seek to exploit.

## Role of Citizens

Citizens can contribute to preventing terrorism by remaining alert, reporting suspicious activities to the appropriate authorities, avoiding the spread of misinformation, and supporting unity within their communities. Respect for diversity, tolerance, and peaceful coexistence helps build stronger societies that are more resilient to hatred and violence.

## Conclusion

Terrorism is a serious threat to global peace, security, and human well-being. It affects people regardless of nationality, religion, or culture. Combating terrorism requires not only effective security measures but also international cooperation, education, social inclusion, and a commitment to justice and human rights. By working together to reject violence and promote peace, societies can help build a safer and more harmonious future for everyone.
"""


initial_state = {
    'essay' : essay
}

workflow.invoke(initial_state)

OutputParserException: Failed to parse EvaluationSchema from completion {"feedback": "The language quality of this essay is exceptional. The writing is remarkably clear, concise, and coherent, making the complex topic of terrorism easily understandable. The vocabulary used is precise and appropriate for an academic discussion, employing terms like \"intimidation,\" \"ideological objectives,\" \"extremist ideologies,\" and \"resilient\" effectively. Sentence structures are varied and grammatically flawless, contributing to a smooth and engaging reading experience. There are no noticeable errors in grammar, spelling, or punctuation. The tone is consistently formal and objective, which is highly suitable for the subject matter. The essay demonstrates a strong command of English, presenting its arguments with professionalism and clarity throughout.", "score": 10}. Got: 1 validation error for EvaluationSchema
score
  Input should be less than 10 [type=less_than, input_value=10, input_type=i

In [ ]:
# essay2 = """# **Terrorism: The Greatest Threat to Peace, Security, and Human Development**

# ## Introduction

# The twenty-first century has witnessed unprecedented advances in science, technology, globalization, and economic development. Yet, alongside these achievements, humanity continues to grapple with one of its most formidable challenges—terrorism. Terrorism transcends geographical boundaries, political systems, and cultural identities. It aims not merely to inflict physical destruction but also to spread fear, weaken democratic institutions, disrupt economies, and divide societies.

# The United Nations defines terrorism broadly as criminal acts intended to provoke a state of terror among the public for political, ideological, or religious purposes. Terrorism is neither confined to any religion nor any particular region; it is a manifestation of violent extremism that threatens the very foundations of peace, justice, and human civilization.

# ## Understanding the Nature of Terrorism

# Terrorism has evolved significantly over the decades. Traditional terrorist organizations relied on localized attacks and limited resources. In contrast, contemporary terrorist groups leverage globalization, encrypted communication, cyber technologies, and social media to recruit followers, raise funds, spread propaganda, and coordinate attacks across borders.

# Modern terrorism includes various forms such as religious extremism, separatist violence, ideological extremism, cyber terrorism, lone-wolf attacks, and transnational terrorism. The emergence of sophisticated technologies has further complicated counter-terrorism efforts, making it a multidimensional challenge.

# ## Causes of Terrorism

# The roots of terrorism are complex and multifaceted. Political instability, prolonged conflicts, weak governance, and unresolved territorial disputes often create conditions that extremist groups exploit. Socio-economic deprivation, unemployment, inequality, and lack of educational opportunities may make vulnerable populations more susceptible to radicalization.

# Extremist ideologies, whether religious, political, or ethnic, provide justification for violence and intolerance. The misuse of digital platforms enables rapid dissemination of propaganda and recruitment. External support, illegal financing, organized crime, and cross-border networks further strengthen terrorist organizations.

# It is important to recognize that while these factors may contribute to the emergence of terrorism, they never justify violence against innocent civilians.

# ## Impact on National and Global Security

# Terrorism poses a direct threat to national sovereignty and global stability. Terrorist attacks result in the tragic loss of innocent lives, destruction of infrastructure, and immense psychological trauma. They create an atmosphere of fear that affects daily life, tourism, investment, and economic growth.

# The financial burden of strengthening security, intelligence gathering, disaster response, and rehabilitation is substantial. Terrorism also weakens investor confidence and disrupts development priorities.

# Beyond physical destruction, terrorism seeks to polarize societies by exploiting religious, ethnic, or ideological differences. Such polarization undermines social harmony, democratic values, and constitutional principles.

# ## Terrorism and Human Rights

# Counter-terrorism efforts present a significant challenge for democratic societies. Governments must protect citizens while safeguarding fundamental rights and the rule of law. Excessive use of force, arbitrary detention, discrimination, or violations of civil liberties may erode public trust and unintentionally contribute to further alienation.

# An effective response to terrorism therefore requires adherence to constitutional values, judicial oversight, accountability, and respect for human dignity.

# ## India's Experience with Terrorism

# India has faced multiple forms of terrorism, including cross-border terrorism, insurgency, left-wing extremism, and isolated acts of violent extremism. Incidents such as the 2001 Parliament attack, the 2008 Mumbai attacks, and other attacks have demonstrated the devastating impact of terrorism on national security and public confidence.

# India has responded through strengthening intelligence coordination, enhancing border management, modernizing security forces, improving financial surveillance, and increasing international cooperation. Legislative measures, technological advancements, and community participation have also become integral components of India's counter-terrorism strategy.

# ## International Cooperation Against Terrorism

# Since terrorism operates across national boundaries, no country can combat it alone. International cooperation is essential in intelligence sharing, financial monitoring, cyber security, extradition arrangements, and capacity building.

# Organizations such as the United Nations encourage member states to cooperate in preventing terrorism while respecting international law and human rights. Global efforts to disrupt terrorist financing, counter online radicalization, and strengthen border security are critical components of a comprehensive strategy.

# ## The Way Forward

# An effective counter-terrorism strategy must extend beyond military and law enforcement responses. It should include:

# * Strengthening intelligence and inter-agency coordination.
# * Enhancing cyber security and technological capabilities.
# * Preventing radicalization through education, awareness, and community engagement.
# * Promoting inclusive economic development and social justice.
# * Strengthening international cooperation against terrorist financing and transnational networks.
# * Upholding constitutional values, human rights, and the rule of law.
# * Encouraging responsible media reporting and countering misinformation.

# Ultimately, defeating terrorism requires addressing both its immediate manifestations and the underlying conditions that extremist groups seek to exploit.

# ## Conclusion

# Terrorism represents one of the gravest threats to humanity because it attacks not only individuals but also the values of peace, democracy, justice, and coexistence. While robust security measures are indispensable, lasting success depends upon inclusive governance, international cooperation, education, social harmony, and respect for human rights.

# As Mahatma Gandhi observed, **"An eye for an eye ends up making the whole world blind."** Sustainable peace cannot be achieved through hatred and violence but through justice, dialogue, and collective determination. The fight against terrorism is therefore not merely a security challenge—it is a moral, political, and developmental responsibility shared by the entire global community.
# """

# initial_state2 = {
#     'essay' : essay2
# }

# workflow.invoke(initial_state2)

